In [1]:
def get_document_type(filename: str) -> str:
    """
    Determine the document type based on the file name pattern.
    
    Patterns:
        claim_PT_15075          -> claim
        discharge_PT_15075      -> discharge
        id_PT_15075             -> id
        amend_PT_15075          -> amendments
        rx_PT_15075             -> prescriptions
        unknown_bank_statement_001 -> unknown
    
    Args:
        filename: The file name (with or without extension) to parse.
    
    Returns:
        A string representing the document type.
    """
    # Strip extension if present
    name = filename.rsplit(".", 1)[0] if "." in filename else filename
    name_lower = name.lower()

    # Mapping of prefixes to document types
    prefix_map = {
        "claim": "CLAIM_FORM",
        "discharge": "DISCHARGE_FORM",
        "id": "ID_DOCUMENT",
        "amend": "AMENDMENTS",
        "rx": "PRESCRIPTIONS",
    }

    # Check known prefixes
    for prefix, doc_type in prefix_map.items():
        if name_lower.startswith(prefix + "_"):
            return doc_type

    # Anything else is unknown
    return "unknown"

In [2]:
from pathlib import Path
import os

def process_file(file_path: str) -> dict:
    path = Path(file_path)
    return {
        "filename": path.name,
        "document_type": get_document_type(path.name),
        "parent_folder": str(path.parent),
        "parent_name": path.parent.name,
        "grandparent": str(path.parent.parent),
    }




#print(process_file("/"))
# {
#   'filename': 'claim_PT_15075.pdf',
#   'document_type': 'claim',
#   'parent_folder': '/data/claims/2024',
#   'parent_name': '2024',
#   'grandparent': '/data/claims'
# }

In [3]:
# Current working directory (usually the notebook's folder in Jupyter)
current_dir = Path.cwd()
BASE_DIR = current_dir.parent.parent
subfolder = BASE_DIR / "dataset/"
# List only directories inside subfolder
folders = [f for f in subfolder.iterdir() if f.is_dir()]

for folder in folders:
    print(folder.name)

claim_forms
discharge_summaries
id_documents
policies
policy_amendments
prescriptions
unknown


In [15]:

subfolder = BASE_DIR / "dataset/id_documents/"
file = next(subfolder.iterdir())
print(subfolder)
print(subfolder.joinpath(file.name))
print(process_file(subfolder.joinpath(file.name)).get("document_type"))


d:\Project\AI\Project\dataset\id_documents
d:\Project\AI\Project\dataset\id_documents\id_PT_15075.png
ID_DOCUMENT


In [4]:
import os
from dotenv import load_dotenv
# Load variables from .env into environment
load_dotenv()

# Read the key
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

In [5]:
import base64
from pathlib import Path
import mimetypes

def image_to_data_url(path: Path) -> str:
    """Convert a local image file to a base64 data URL."""
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Image not found: {path}")

    # Detect MIME type (jpeg, png, webp, gif)
    mime, _ = mimetypes.guess_type(path.name)
    if mime is None or not mime.startswith("image/"):
        raise ValueError(f"Unsupported image type: {path.name}")

    with path.open("rb") as f:
        encoded = base64.b64encode(f.read()).decode("utf-8")

    return f"data:{mime};base64,{encoded}"

In [8]:
subfolder = BASE_DIR / "dataset/unknown/"
file = next(subfolder.iterdir())
file

WindowsPath('d:/Project/AI/Project/dataset/unknown/unknown_bank_statement_001.png')

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate

# 1. Define your structured output schema
class DocumentClassification(BaseModel):
    doc_type: Literal[
        "CLAIM_FORM", "ID_DOCUMENT", "DISCHARGE_SUMMARY", 
        "PRESCRIPTION", "POLICY_AMENDMENT", "UNKNOWN"
    ] = Field(description="The classified type of the document")
    confidence: float = Field(description="Confidence score between 0.0 and 1.0", ge=0, le=1)
    routing_tags: list[str] = Field(description="Tags for routing the document")

# 2. Initialize the Groq vision model
# Note: Use the specific vision model ID
llm = ChatGroq(
    model=os.getenv("MODEL_NAME"),
    temperature=0, # Keep it deterministic for classification
    api_key=GROQ_API_KEY
)

# 3. Bind the schema to the LLM for structured output
structured_llm = llm.with_structured_output(DocumentClassification)

# 4. Create your prompt and chain
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a document classification assistant. Analyze the image and classify it."),
    ("human", [
        {"type": "text", "text": "Classify the following document."},
        {"type": "image_url", "image_url": {"url": "{image_url}"}}
    ])
])

classifier_chain = prompt | structured_llm

#5. Invoke the chain (example)
image_path = subfolder.joinpath(file.name)
data_url = image_to_data_url(image_path)
result = classifier_chain.invoke({"image_url": data_url})
print(result) # Returns a DocumentClassification Pydantic object

#6. Your human review queue logic (conceptual)
if result.confidence < 0.7 or result.doc_type == "UNKNOWN":
    print("Route to human review:", result)

In [7]:
import pytesseract
from pytesseract import Output
from PIL import Image

def tesseract_structured(image_path,doctment_name):
    image = Image.open(image_path)
    data = pytesseract.image_to_data(image, output_type=Output.DICT)
    
    fields = []
    all_text = []
    bounding_boxes = []
    
    for i in range(len(data['text'])):
        # Filter out empty entries (Tesseract returns many blank rows)
        if data['text'][i].strip():
            # Build the field entry
            fields.append({
                'text': data['text'][i],
                'confidence': int(data['conf'][i]),
                'bbox': (
                    data['left'][i],
                    data['top'][i],
                    data['left'][i] + data['width'][i],
                    data['top'][i] + data['height'][i]
                )
            })
            
            all_text.append(data['text'][i])
            bounding_boxes.append((
                data['left'][i],
                data['top'][i],
                data['width'][i],
                data['height'][i]
            ))
    
    return {
        'document_name': doctment_name,
        'fields': fields,
        'raw_text': ' '.join(all_text),
        'bounding_boxes': bounding_boxes
    }


In [8]:
parent_folder = BASE_DIR / "dataset/"

# List only directories inside subfolder
sub_folders = [f for f in parent_folder.iterdir() if f.is_dir()]
doc = dict()
for folder in sub_folders:
    subfolder = parent_folder.joinpath(folder)
    iterator = list(subfolder.iterdir())
    first_two = iterator[0:1]   # TypeError!
    ext = os.path.splitext(first_two[0])[1][1:]
    if ext.lower() in ['jpg', 'jpeg', 'png', 'tiff', 'bmp', 'gif']:    
        result = tesseract_structured(first_two[0], get_document_type(first_two[0].name))
        doc[result['document_name']] = result['raw_text']



In [9]:
temp = doc.copy()
for key in ["AMENDMENTS", "PRESCRIPTIONS", "unknown"]:
    temp.pop(key, None)

In [ ]:
import os
import json
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain.agents import create_agent
import re

load_dotenv()


# ---------- TOOL ----------
@tool
def extract_name(text: str) -> str:
    """Extract the person's full name from the given text.
    Look for labels like NAME, PATIENT NAME, MEMBER NAME, INSURED NAME."""
    import re
    m = re.search(
        r"(?:NAME|PATIENT\s*NAME|MEMBER\s*NAME|INSURED\s*NAME)"
        r"\s*[:\-]?\s*([^\n\r:]+)",
        text, re.IGNORECASE,
    )
    if m:
        return m.group(1).strip().rstrip(".,;:")
    return "NOT_FOUND"

@tool
def extract_birth_date(claim_text: str) -> str:
    """Extract the patient's birth date from a CLAIM form.
    Recognizes: PATIENT'S BIRTH DATE, BIRTH DATE, DOB, D.O.B, DATE OF BIRTH.
    Returns JSON: {"birth_date": "..."} or {"birth_date": "NOT_FOUND"}."""

    # Label variants
    label = (
        r"(?:PATIENT'?S?\s*)?"
        r"(?:BIRTH\s*DATE|DATE\s*OF\s*BIRTH|D\.?O\.?B\.?)"
    )

    # Accepted date formats
    #   1985-03-15 | 15/03/1985 | 03/15/1985 | 15-03-1985
    #   15-Mar-1985 | 15 Mar 1985 | March 15, 1985
    date_val = (
        r"("
        r"\d{4}[-/]\d{1,2}[-/]\d{1,2}"
        r"|\d{1,2}[-/]\d{1,2}[-/]\d{2,4}"
        r"|\d{1,2}[-/ ][A-Za-z]{3,9}[-/ ]\d{2,4}"
        r"|[A-Za-z]{3,9}\s+\d{1,2},?\s+\d{2,4}"
        r")"
    )

    m = re.search(label + r"\s*[:\-]?\s*" + date_val, claim_text, re.IGNORECASE)
    if m:
        return json.dumps({"birth_date": m.group(1).strip()})
    return json.dumps({"birth_date": "NOT_FOUND"})



@tool
def extract_insured_id(claim_text: str) -> str:
    """Extract the insured ID / policy number from a CLAIM form. Start with MED-GLD-
    Recognizes: INSURED'S I.D. NUMBER,INSURED'S I.D.NUMBER, INSURED ID NO, INSURED UNIQUE ID
    POLICY NUMBER, MEMBER NUMBER, MEMBER ID,MEMBER I.
    Returns JSON: {"insured_id": "..."} or {"insured_id": "NOT_FOUND"}."""

    patterns = [
        # INSURED'S I.D. NUMBER  |  INSURED ID NO  |  INSURED ID #
        r"INSURED'?S?\s*I\.?D\.?\s*(?:NUMBER|NO\.?|#)?",
        # INSURED UNIQUE ID
        r"INSURED\s*UNIQUE\s*ID",
        # POLICY NUMBER  |  POLICY NO  |  POLICY #
        r"POLICY\s*(?:NUMBER|NO\.?|#)",
        # MEMBER NUMBER  |  MEMBER ID
        r"MEMBER\s*(?:NUMBER|I\.?D\.?|NO\.?|#)",
    ]

    for label in patterns:
        m = re.search(
            label + r"\s*[:\-]?\s*([A-Z0-9][A-Z0-9\-/]*)",
            claim_text, re.IGNORECASE,
        )
        if m:
            return json.dumps({"insured_id": m.group(1).strip()})

    return json.dumps({"insured_id": "NOT_FOUND"})

# ---------- AGENT ----------
def build_agent():
    llm = ChatGroq(
        model="openai/gpt-oss-120b",
        temperature=0,
        api_key=os.getenv("GROQ_API_KEY"),
    )
    return create_agent(
        model=llm,
        tools=[extract_name, extract_birth_date, extract_insured_id],
        system_prompt=(
            "You extract information from CLAIM forms. When given text, call the appropriate tool once "
            "and reply with ONLY the extracted information as plain text. If not found, reply NOT_FOUND."
        ),
    )


# ---------- PUBLIC API ----------
def get_name(text: str) -> str:
    agent = build_agent()
    result = agent.invoke({"messages": [{"role": "user", "content": text}]})
    return result["messages"][-1].content.strip()




In [16]:
print(get_name(temp['ID_DOCUMENT']))

John Johnsor


In [12]:
temp['ID_DOCUMENT']

'MediShield Health Insurance Ltd. MED-GLD | Group: GRP-41029, Member Name: John Johnsor Member I MED-GLD-3592974 Group Number: GRP-41029 Effective Date: 07/01/2025 Expiration Da * CARD EXPIRED — NOT VALID ** PCP Visit: Specialist Emergency Room Urgent Care: Deductible (Ind.): Out-of- Pocket Max’ Rx Tier 1 (Generic): $25 copay $60 copay $250 copay $50 copay $1,500 $7,500 $15 / 30-day'